<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">DISTRIBUTED LORA FINE-TUNING WITH LLAMAFACTORY (HEAD)</h2>

<p>LoRA is a lightweight fine-tuning method for large language models that injects small, trainable adapter layers into the model instead of updating all parameters. This makes training far more memory and compute-efficient, while reducing the risk of significant forgetting compared to full fine-tuning. However, LoRA adapters may sometimes trade off a bit of peak accuracy for efficiency and flexibility.</p>

<p>Check out this article for more information: 
<a href="https://medium.com/@kailash.thiyagarajan/fine-tuning-large-language-models-with-lora-demystifying-efficient-adaptation-25fa0a389075">
Fine-Tuning Large Language Models with LORA: Demystifying Efficient Adaptation</a></p>

<!-- Modules Overview -->
<div style="border-left:4px solid #44D62C; padding:12px 16px; background-color:transparent; margin:14px 0; font-size:14px; border-radius:6px;">
  <b style="color:#44D62C;">Note:</b>  
  This guide uses the following key modules and tools:
  <ul style="margin:8px 0 0 18px;">
    <li><code>llamafactory-cli</code> – streamlined CLI for fine-tuning with optimized defaults and configuration management.</li>
    <li><code>rzr-aikit</code> – Razer CLI to download models, serve fine-tuned models with LoRA adapters, run generation tests and control model deployment.</li>
  </ul>
  <b style="color:#44D62C;">Dependencies:</b> LlamaFactory internally uses <code>transformers</code> and <code>datasets</code> for model handling and data processing.
</div>

<h3 style="color:#44D62C; text-align:left;">📥 1. Download the Model</h3>

Before running a model in distributed mode, ensure it is downloaded on every participating machine.

This command will pull the model from Hugging Face into the local Razer environment cache.

In [ ]:
rzr-aikit model download Qwen/Qwen2.5-0.5B-Instruct

<h3 style="color:#44D62C; text-align:left;">🔗 2. Start Cluster on Head Node</h3>

Next, start the cluster on your **head node**.

This node will coordinate scheduling, memory partitioning, and request routing across the cluster.

Use the `--ifname` flag to specify the network interface used for inter-node communication (e.g. `enp47s0`). You can identify it with `ip` or `ifconfig`.

In [ ]:
ip -f inet -4 -o addr

In [ ]:

rzr-aikit cluster run --ifname enp47s0 --metrics-export-port 8080

<h3 style="color:#44D62C; text-align:left;">🛰️ 3. Check Cluster Status</h3>

Verify that the cluster head node is running and ready to accept worker connections.

For distributed model deployment, at least <strong>two active nodes</strong> are required — one head and at least one worker node.

In [ ]:
rzr-aikit cluster status

If no worker nodes are shown, follow Guide <strong>4b_(Node)_Distributed_Fine_Tuning_LoRA.ipynb<strong>

<h3 style="color:#44D62C; text-align:left;">📊 4. Dataset Configuration</h3>

LlamaFactory uses JSON configuration files to define dataset parameters, including data sources, formatting templates, and column mappings. This approach provides flexibility in handling various dataset formats and ensures consistent data preprocessing across distributed training.

In [ ]:
# Create data directory and dataset configuration
mkdir -p ~/fine-tuning/data
cat > ~/fine-tuning/data/dataset_info.json << 'JSON'
{
  "alpaca_gpt4": {
    "hf_hub_url": "vicgalle/alpaca-gpt4",
    "formatting": "alpaca",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}
JSON

<h3 style="color:#44D62C; text-align:left;">⚙️ 5. LlamaFactory YAML Configuration</h3>

Instead of using long command-line arguments, we'll create a YAML configuration file for LlamaFactory. This approach provides better reproducibility, version control, and easier parameter management for distributed training. Note that some configurations like Ray are only available in YAML format and cannot be specified through command-line arguments.

In [ ]:
gpu-discover --distributed

<!-- Ray Configuration Note -->
<div style="border-left:4px solid #44D62C; padding:12px 16px; background-color:transparent; margin:14px 0; font-size:14px; border-radius:6px;">
  <b style="color:#44D62C;">Configuration Note:</b>  
  Modify <code>ray_num_workers: 1</code> based on the number of GPUs you want to use for distributed fine-tuning.
</div>

In [ ]:
mkdir -p ~/fine-tuning/configs
cat > ~/fine-tuning/configs/lora_distributed_training.yaml << 'YAML'
### Model Configuration
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
template: qwen

### Training Configuration
stage: sft
do_train: true
finetuning_type: lora

### LoRA Parameters
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target: q_proj,k_proj,v_proj,o_proj

### Dataset and Output Configuration
dataset: alpaca_gpt4
dataset_dir: /home/Razer/fine-tuning/data
output_dir: ~/fine-tuning/adapters/lora_alpaca_gpt4_distributed
overwrite_output_dir: true

### Training Hyperparameters
num_train_epochs: 1
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2e-4
warmup_ratio: 0.03
cutoff_len: 768
packing: true
max_samples: 2000

### Logging and Evaluation
eval_strategy: "no"
logging_strategy: steps
logging_steps: 50
save_strategy: "no"

### Memory and Performance Optimizations
gradient_checkpointing: true
group_by_length: true
fp16: true

### Ray Distributed Training Configuration
ray_run_name: qwen2p5_lora_distributed
ray_num_workers: 4  # Set this to the number of GPUs you want to use for training
ray_storage_path: ~/fine-tuning/
placement_strategy: PACK
resources_per_worker:
  GPU: 1
YAML

<h3 style="color:#44D62C; text-align:left;">🚀 6. Execute Distributed LoRA Fine-Tuning</h3>

Now we'll run the distributed fine-tuning using Ray backend with our YAML configuration. The training will automatically utilize Ray workers across the cluster nodes, with each worker using 1 GPU as specified in the configuration.

In [ ]:
export USE_RAY=1
llamafactory-cli train ~/fine-tuning/configs/lora_distributed_training.yaml

<h3 style="color:#44D62C; text-align:left;">🔄 7. Serve Fine-Tuned Model with vLLM</h3>

Use VLLM to serve the base model with integrated LoRA adapters for high-performance inference. The server supports multiple LoRA adapters simultaneously, allowing dynamic model switching during inference.

In [ ]:
nohup vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --enable-lora \
  --lora-modules alpaca=~/fine-tuning/adapters/lora_alpaca_gpt4_distributed \
  --gpu-memory-utilization 0.7 \
  --max-model-len 2048 \
  --enforce-eager \
  > ~/vllm.log 2>&1 &

In [ ]:
tail -f ~/vllm.log \
 | sed -n 'p;/Application startup complete\./q'

<h3 style="color:#44D62C; text-align:left;">🧪 8. Test Base Model Performance</h3>

Generate responses using the base model (without LoRA adapters) to establish baseline performance for comparison with the fine-tuned version.

In [ ]:
rzr-aikit model generate --model Qwen/Qwen2.5-0.5B-Instruct "Explain the concept of machine learning in simple terms."

<h3 style="color:#44D62C; text-align:left;">✨ 9. Test Fine-Tuned Model Performance</h3>

Generate responses using the fine-tuned LoRA adapter to evaluate the impact of instruction-following training on model behavior and response quality.

In [ ]:
rzr-aikit model generate --model alpaca "Explain the concept of machine learning in simple terms."

<h3 style="color:#44D62C; text-align:left;">🛑 10. Cleanup</h3>

Once testing is complete, stop the model inference process to free up system resources across all cluster nodes.

In [ ]:
rzr-aikit model stop

In [ ]:
rzr-aikit cluster stop